# `MatrixVisualizer` — интерактивная heatmap матрицы признаков

Класс для интерактивной визуализации матрицы `N_features × K` (например, агрегатов признаков по времени —
для анализа дрифта) в виде heatmap на **Bokeh**. Возможности:

- покраска по квантилям (`QuantileTransformer`), палитра `coolwarm`, colorbar с подписями;
- переключение оси покраски на лету (по строкам / по столбцам / глобально) через выпадающий список;
- строки отсортированы кластеризацией **HDBSCAN**, так что похожие признаки стоят рядом;
- зум, при котором подписи признаков появляются только на достаточном приближении (читаемость при больших `N`);
- выделение признаков (Tap / Box-select) с выводом их имён в текстовое поле и кнопкой «Копировать» —
  удобно, чтобы быстро собрать список «плохих» признаков для выкидывания.

Весь интерактив — на python-коллбэках через локальный Bokeh-сервер в ноутбуке (JS только для копирования в буфер).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import sklearn
import bokeh
import warnings

from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import HDBSCAN
from sklearn.datasets import load_breast_cancer

from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, Select, LinearColorMapper, ColorBar, CustomJSTickFormatter, CustomJS, TextAreaInput, Button
from bokeh.layouts import column, row
from bokeh.io import show, output_notebook

import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Игнорируем варнинги от QuantileTransformer, когда n_quantiles > n_samples
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.preprocessing')

### Данные для демонстрации
Синтетическая матрица с разными паттернами дрифта и реальная матрица (breast cancer). В синтетической матрицы есть паттерны по строкам, но так как строки перемешаны без кластеризации они не были бы визуально заметны

In [7]:
def generate_mock_drift_data(n_features=500, n_days=365):
    dates = pd.date_range(start="2024-01-01", periods=n_days, freq='7D').strftime('%Y-%m-%d').tolist()
    X = np.zeros((n_features, n_days))

    X[0:100, :] = np.random.normal(0, 1, (100, n_days))

    X[100:200, :] = np.random.normal(0, 1, (100, n_days))
    X[100:200, int(n_days * 0.4):int(n_days * 0.6)] += 5  # Горб в середине

    X[200:300, :] = np.random.normal(0, 1, (100, n_days))
    X[200:300, int(n_days * 0.7):] -= 5                 # Падение в конце

    X[300:400, ::int(n_days * 0.05)] += 3               # Периодичность
    X[300:400, :] += np.random.normal(0, 0.5, (100, n_days))

    X[400:500, :] = np.random.normal(0, 10, (100, n_days))

    np.random.shuffle(X)

    feature_names = [f"feature_{i:03d}" for i in range(n_features)]
    df = pd.DataFrame(X, index=feature_names, columns=dates)

    return df

df_mock = generate_mock_drift_data()
print(f"Размер матрицы: {df_mock.shape}")
df_mock.head()

Размер матрицы: (500, 365)


,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,...,2030-10-21,2030-10-28,2030-11-04,2030-11-11,2030-11-18,2030-11-25,2030-12-02,2030-12-09,2030-12-16,2030-12-23
feature_000,5.459739,9.722055,-6.615326,12.339168,-0.184333,-6.444544,0.173673,11.958233,8.322397,18.461696,...,-0.116774,1.636276,5.391417,-0.161050,-0.504766,-9.087573,-15.148391,9.659238,4.011737,-10.813292
feature_001,-23.208141,8.943466,4.153530,8.105576,7.380283,5.175996,-4.622226,3.810783,23.675225,4.227800,...,0.173417,-4.541882,-5.669019,-3.328285,-16.154378,-6.252854,-5.704581,1.897227,-15.493101,9.338208
feature_002,3.117020,0.041153,-0.548698,0.110514,-0.722391,-1.060751,-0.219737,0.077666,-0.562809,0.556624,...,1.574095,0.285560,-0.426980,-0.066212,-0.169860,3.210285,-0.500816,-0.362125,0.314610,0.186688
feature_003,15.889913,-0.567123,-4.553706,-2.086994,-3.105318,7.876706,-14.261494,8.520577,-6.116604,-11.096346,...,-9.230320,10.326335,-8.998305,-2.268017,11.467117,1.141970,0.613610,-4.164514,2.321546,-8.615811
feature_004,0.093159,-0.403830,-0.277098,-0.339847,0.371777,-0.047698,0.256754,1.024773,0.461535,-0.079443,...,-3.464886,-7.185997,-4.407275,-5.814976,-4.905894,-5.527202,-5.787509,-5.348436,-4.769403,-5.875856


In [3]:
cancer = load_breast_cancer()
df_real = pd.DataFrame(
    cancer.data.T,
    index=cancer.feature_names,
    columns=[str(i) for i in range(cancer.data.shape[0])]
)

df_real = df_real.iloc[:, :150]

print(f"Размер реальной матрицы: {df_real.shape}")
df_real.head()

Размер реальной матрицы: (30, 150)


,0,1,2,3,4,5,6,7,8,9,...,140,141,142,143,144,145,146,147,148,149
mean radius,17.9900,20.57000,19.6900,11.4200,20.2900,12.4500,18.25000,13.7100,13.0000,12.4600,...,9.7380,16.11000,11.4300,12.90000,10.75000,11.9000,11.8000,14.95000,14.4400,13.74000
mean texture,10.3800,17.77000,21.2500,20.3800,14.3400,15.7000,19.98000,20.8300,21.8200,24.0400,...,11.9700,18.05000,17.3100,15.92000,14.97000,14.6500,16.5800,18.77000,15.1800,17.91000
mean perimeter,122.8000,132.90000,130.0000,77.5800,135.1000,82.5700,119.60000,90.2000,87.5000,83.9700,...,61.2400,105.10000,73.6600,83.74000,68.26000,78.1100,78.9900,97.84000,93.9700,88.12000
mean area,1001.0000,1326.00000,1203.0000,386.1000,1297.0000,477.1000,1040.00000,577.9000,519.8000,475.9000,...,288.5000,813.00000,398.0000,512.20000,355.30000,432.8000,432.0000,689.50000,640.1000,585.00000
mean smoothness,0.1184,0.08474,0.1096,0.1425,0.1003,0.1278,0.09463,0.1189,0.1273,0.1186,...,0.0925,0.09721,0.1092,0.08677,0.07793,0.1152,0.1091,0.08138,0.0997,0.07944


In [4]:
class MatrixVisualizer:
    """
    Класс для интерактивной визуализации матрицы признаков с помощью Bokeh.
    Реализует квантильное преобразование, кластеризацию строк (HDBSCAN) и анализ дрифта.
    """
    def __init__(self, df: pd.DataFrame, min_cluster_size: int = 5):
        self.df_original = df.copy()
        # Bokeh требует строковые значения для категориальных осей
        self.time_steps = df.columns.astype(str).tolist()
        self.min_cluster_size = min_cluster_size

        self.df_colored = self._transform_data(self.df_original, axis='rows')
        self._cluster_and_sort()

    def _transform_data(self, df: pd.DataFrame, axis: str = 'rows') -> pd.DataFrame:
        qt = QuantileTransformer(n_quantiles=6)

        if axis == 'rows':
            transformed = qt.fit_transform(df.T).T
        elif axis == 'columns':
            transformed = qt.fit_transform(df)
        elif axis == 'global':
            flat_data = df.values.reshape(-1, 1)
            transformed = qt.fit_transform(flat_data).reshape(df.shape)
        else:
            raise ValueError("Параметр axis должен быть 'rows', 'columns' или 'global'")

        return pd.DataFrame(transformed, index=df.index, columns=df.columns.astype(str))

    def _cluster_and_sort(self):
        clusterer = HDBSCAN(min_cluster_size=self.min_cluster_size, copy=True)
        cluster_labels = clusterer.fit_predict(self.df_colored.values)

        self.df_colored['cluster_label'] = cluster_labels
        self.df_original['cluster_label'] = cluster_labels

        self.df_colored = self.df_colored.sort_values(by='cluster_label')
        self.df_original = self.df_original.sort_values(by='cluster_label')

        self.df_colored.drop(columns=['cluster_label'], inplace=True)
        self.df_original.drop(columns=['cluster_label'], inplace=True)

        self.feature_names = self.df_colored.index.tolist()

    def _get_long_format(self, axis: str) -> pd.DataFrame:
        df_transformed = self._transform_data(self.df_original, axis=axis)
        df_transformed = df_transformed.loc[self.feature_names]

        df_melted = df_transformed.reset_index().melt(
            id_vars=df_transformed.index.name or 'index',
            var_name='time',
            value_name='value'
        )
        df_melted.columns = ['feature', 'time', 'value']
        return df_melted

    def modify_doc(self, doc):
        df_melted = self._get_long_format('rows')

        time_to_idx = {ts: i for i, ts in enumerate(self.time_steps)}
        df_melted['time_idx'] = df_melted['time'].map(time_to_idx)

        source = ColumnDataSource(df_melted)

        coolwarm_palette = [mcolors.to_hex(c) for c in cm.coolwarm(np.linspace(0, 1, 256))]
        mapper = LinearColorMapper(palette=coolwarm_palette, low=0, high=1)

        p = figure(
            width=1000, height=600,
            x_range=(-0.5, len(self.time_steps) - 0.5),
            y_range=self.feature_names,
            tools="pan,wheel_zoom,box_zoom,reset,tap,box_select",
            toolbar_location="right",
            active_scroll="wheel_zoom",
            x_axis_location="above"
        )

        p.xaxis.formatter = CustomJSTickFormatter(
            args=dict(dates=self.time_steps),
            code="""
                var idx = Math.round(tick);
                if (idx >= 0 && idx < dates.length) {
                    return dates[idx];
                }
                return '';
            """
        )

        p.xaxis.major_label_orientation = np.pi / 4
        p.xaxis.major_label_text_align = "left"
        p.xaxis.major_label_text_baseline = "middle"

        p.yaxis.major_tick_line_color = None
        p.yaxis.minor_tick_line_color = None
        p.yaxis[0].major_label_text_font_size = '0pt'

        def update_labels_visibility(attr, old, new):
            if p.y_range.start is not None and p.y_range.end is not None:
                visible_count = p.y_range.end - p.y_range.start
                if visible_count <= 50:
                    p.yaxis[0].major_label_text_font_size = '10pt'
                else:
                    p.yaxis[0].major_label_text_font_size = '0pt'

        p.y_range.on_change('start', update_labels_visibility)
        p.y_range.on_change('end', update_labels_visibility)

        p.rect(
            x='time_idx', y='feature', width=1, height=1,
            source=source,
            fill_color={'field': 'value', 'transform': mapper},
            line_color=None
        )

        color_bar = ColorBar(color_mapper=mapper, location=(0, 0))
        p.add_layout(color_bar, 'right')

        axis_select = Select(value='rows', options=['rows', 'columns', 'global'], title="Ось покраски:")

        def update_plot(attr, old, new):
            new_df = self._get_long_format(new)
            new_df['time_idx'] = new_df['time'].map(time_to_idx)
            source.data = dict(ColumnDataSource(new_df).data)

        axis_select.on_change('value', update_plot)

        text_area = TextAreaInput(title="Выбранные признаки:", value="", rows=3, width=800)
        copy_button = Button(label="Копировать", button_type="success", width=150)

        def update_selection(attr, old, new):
            indices = source.selected.indices
            if indices:
                selected_features = sorted(list(set([source.data['feature'][i] for i in indices])))
                text_area.value = ", ".join([f"'{f}'" for f in selected_features])
            else:
                text_area.value = ""

        source.selected.on_change('indices', update_selection)

        copy_callback = CustomJS(args=dict(text_area=text_area), code="""
            navigator.clipboard.writeText(text_area.value).then(function() {
                console.log('Copied to clipboard!');
            }, function(err) {
                console.error('Could not copy text: ', err);
            });
        """)
        copy_button.js_on_click(copy_callback)

        bottom_row = row(text_area, copy_button)
        layout = column(axis_select, p, bottom_row)
        doc.add_root(layout)

### Запуск

In [5]:
output_notebook()

os.environ["BOKEH_ALLOW_WS_ORIGIN"] = "*"

visualizer_mock = MatrixVisualizer(df_mock)
show(visualizer_mock.modify_doc)

Loading BokehJS ...

In [6]:
visualizer_real = MatrixVisualizer(df_real)
show(visualizer_real.modify_doc)